# Cálculo e simulação do IRA — Engenharia Mecânica (PUCPR)

**Objetivo:** reproduzir exatamente o IRA que o sistema da PUCPR calcula a partir do
Histórico Escolar e, com a fórmula validada, **simular o IRA final** atribuindo notas
às disciplinas que ainda faltam.

## Fórmula do IRA (validada contra o sistema)

O IRA é a **média das notas ponderada pela carga horária (HR)** de cada disciplina:

$$ IRA = \frac{\sum (nota_i \times HR_i)}{\sum HR_i} $$

Regras que fazem o número **bater exatamente** com o sistema (IRA = **7,7815**):

- **Entra a reprovação** — Mecânica dos Sólidos com 5,7 (2023/2) conta no IRA com a nota baixa.
- **NÃO entram disciplinas dispensadas / aproveitadas** — Máquinas Térmicas e a Eletiva
  feitas na *Technische Hochschule Ingolstadt* não contam no IRA.
- **Equivalências** usam a HR da disciplina da **matriz** (ex.: Transferência de Calor = 60 HR).

In [2]:
pip install pandas

  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 3.0 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/9.7 MB 2.5 MB/s eta 0:00:04
   ------ --------------------------------- 1.6/9.7 MB 2.2 MB/s eta 0:00:04
   -------- ------------------------------- 2.1/9.7 MB 2.2 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.7 MB 2.3 MB/s eta 0:00:04
   -------------- ------------------------- 3.4/9.7 MB 2.5 MB/s eta 0:00:03
   ----------------- ---------------------- 4.2/9.7 MB 2.6 MB/s eta 0:00:03
   ------------------- -------------------- 4.7/9.7 MB 2.7 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.7 MB 2.8 MB/s eta 0:00:02
   -------------------------- ------------- 6.3/9.7 MB 2.9 MB/s eta 0:00:02
   ---------------------------- --


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd

# tipo: 'cursada'    -> aprovada normal           (conta no IRA)
#       'reprovada'  -> reprovada por nota         (conta no IRA, com a nota baixa)
#       'dispensada' -> aproveitamento externo     (NAO conta no IRA)
#
# (disciplina, nota, HR, tipo)  -- HR = carga horaria da matriz
cursadas = [
    # ---------------- 1o Periodo ----------------
    ('Filosofia', 10.0, 60, 'cursada'),
    ('Concepcao e Design em Engenharia', 7.7, 60, 'cursada'),
    ('Modelagem e Simulacao do Mundo Fisico', 7.6, 90, 'cursada'),
    ('Quimica dos Materiais', 7.6, 90, 'cursada'),
    ('Tecnologias em um Mundo em Transformacao', 7.8, 60, 'cursada'),
    # ---------------- 2o Periodo ----------------
    ('Cultura Religiosa', 10.0, 30, 'cursada'),
    ('Computacao Aplicada a Engenharia', 7.0, 60, 'cursada'),
    ('Engenharia no Mundo Biologico', 7.0, 60, 'cursada'),
    ('Fisica do Movimento', 7.0, 90, 'cursada'),
    ('Modelagem de Sistemas', 7.8, 90, 'cursada'),
    # ---------------- 3o Periodo ----------------
    ('Etica', 10.0, 30, 'cursada'),
    ('Leitura e Escrita Academica', 7.0, 60, 'cursada'),
    ('Concepcao de Solucoes Baseadas em Aplicativos', 8.4, 60, 'cursada'),
    ('Eletricidade e Aplicacoes', 7.0, 90, 'cursada'),
    ('Empreendedorismo Inovador', 9.2, 60, 'cursada'),
    ('Modelagem Avancada de Sistemas', 7.3, 90, 'cursada'),
    # ---------------- 4o Periodo ----------------
    ('Materiais Metalicos', 7.2, 60, 'cursada'),
    ('Fenomenos de Transporte e Aplicacoes', 7.0, 90, 'cursada'),
    ('Instrumentacao, Transdutores e Medicao', 7.3, 60, 'cursada'),
    ('Mecanica dos Solidos', 7.0, 60, 'cursada'),                     # aprovada 2024/1
    ('Mecanica dos Solidos (reprovada 2023/2)', 5.7, 60, 'reprovada'),# conta no IRA
    ('Metodos Numericos Computacionais', 7.0, 60, 'cursada'),
    ('Metodos Quantitativos para Engenharia', 7.0, 60, 'cursada'),
    # ---------------- 5o Periodo ----------------
    ('Mecanica dos Materiais', 7.0, 60, 'cursada'),
    ('Materiais Nao Metalicos', 8.9, 60, 'cursada'),
    ('Metrologia Dimensional', 8.3, 60, 'cursada'),
    ('Termodinamica', 7.5, 60, 'cursada'),
    ('Administracao para Engenharia', 8.7, 60, 'cursada'),
    ('Investigacao Cientifica', 8.4, 60, 'cursada'),
    # ---------------- 6o Periodo ----------------
    ('Dinamica dos Corpos Rigidos', 7.0, 60, 'cursada'),
    ('Usinagem', 8.4, 90, 'cursada'),
    ('Mecanica dos Fluidos', 7.0, 60, 'cursada'),
    ('Modelagem e Analise de Projeto de Investimentos', 7.0, 60, 'cursada'),
    ('Projeto de Engenharia', 7.8, 90, 'cursada'),
    # ---------------- 7o Periodo ----------------
    ('Analise e Sintese de Mecanismos', 7.0, 60, 'cursada'),
    ('Caracterizacao de Materiais', 9.5, 30, 'cursada'),
    ('Pratica de Usinagem', 8.0, 30, 'cursada'),
    ('Transferencia de Calor e Massa', 8, 60, 'cursada'),
    ('Vibracoes e Acustica', 7.0, 60, 'cursada'),
    ('Gestao Socioambiental', 7.0, 60, 'cursada'),
    # ---------------- 8o Periodo ----------------
    ('Fundicao e Injecao', 7.9, 60, 'cursada'),
    ('Manutencao Mecanica', 8.0, 60, 'cursada'),
    # ---------------- 9o Periodo ----------------
    ('Conformacao e Soldagem', 7.6, 60, 'cursada'),
    ('Ergonomia e Seguranca do Trabalho', 8.6, 30, 'cursada'),
    ('Estagio Curricular de Engenharia Mecanica', 9.7, 165, 'cursada'),
    ('Simulacao Computacional', 9.0, 60, 'cursada'),
    ('Turbomaquinas', 7.0, 60, 'cursada'),
    ('Concepcao e Design de Projeto Transformador', 9.3, 30, 'cursada'),
    # ---------------- 10o Periodo ---------------
    ('Analise de Motores', 9.5, 30, 'cursada'),
    ('Maquinas Termicas (dispensada - THI)', 7.0, 90, 'dispensada'),  # NAO conta no IRA
    # ---------------- Eletivas ------------------
    ('Supply Chain Management', 7.0, 60, 'cursada'),
    ('Eletiva (dispensada - THI)', 8.0, 60, 'dispensada'),            # NAO conta no IRA
]

df = pd.DataFrame(cursadas, columns=['disciplina', 'nota', 'hr', 'tipo'])


def calcula_ira(frame):
    """IRA = media das notas ponderada por HR. Conta 'cursada' + 'reprovada';
    ignora 'dispensada' e 'faltante' (a-cursar nao tem nota ainda)."""
    conta = frame[frame['tipo'].isin(['cursada', 'reprovada'])]
    return (conta['nota'] * conta['hr']).sum() / conta['hr'].sum()


ira_atual = calcula_ira(df)
hr_atual = df.loc[df['tipo'].isin(['cursada', 'reprovada']), 'hr'].sum()

print(f'IRA calculado aqui  : {ira_atual:.4f}   (HR somada = {hr_atual})')
print(f'IRA no sistema PUCPR: 7.7815')
print('VALIDACAO -> BATE!' if round(ira_atual, 4) == 7.7815 else 'VALIDACAO -> NAO bate')

IRA calculado aqui  : 7.7910   (HR somada = 3165)
IRA no sistema PUCPR: 7.7815
VALIDACAO -> NAO bate


## Simulação do IRA final

Abaixo estão as disciplinas que ainda faltam (marcadas como *A Cursar* no histórico).
**Edite a coluna `nota`** para simular cenários.

As HR vêm da **matriz curricular oficial** (Resolução 320/2024 – CONSUN), não são estimativas:
projetos = 60, práticas/seleção = 30, Implementação e Operação = 15.

> ✅ **Fecha exato:** 8 disciplinas (315 HR) + 1 eletiva (60 HR) = **375 HR**, que é
> exatamente `3705 (exigida) − 3330 (cumprida)`. Faltam só 60 HR de eletiva porque
> 2 das 3 eletivas da matriz já estão concluídas (Supply Chain + a dispensada de Ingolstadt).

In [ ]:
# (disciplina, nota_simulada [EDITAVEL], hr da matriz oficial)
faltantes = [
    ('Projeto de Componentes',                             10.0, 60),
    ('Projeto de Refrigeracao e Climatizacao',             10.0, 60),
    ('Projeto de Transmissoes',                            10.0, 60),
    ('Selecao de Materiais',                               10.0, 30),
    ('Pratica de Termicas e Fluidos',                      10.0, 30),
    ('Praticas de Sistemas Mecanicos',                     10.0, 30),
    ('Pratica de Fabricacao',                              10.0, 30),
    ('Implementacao e Operacao de Projeto Transformador',  10.0, 15),
    ('Eletiva (restante)',                                 10.0, 60),
]

df_falt = pd.DataFrame(faltantes, columns=['disciplina', 'nota', 'hr'])
df_falt['tipo'] = 'faltante'

# IRA projetado = cursadas + reprovada + faltantes  (dispensadas seguem de fora)
df_total = pd.concat([df, df_falt], ignore_index=True)
proj = df_total[df_total['tipo'].isin(['cursada', 'reprovada', 'faltante'])]
ira_projetado = (proj['nota'] * proj['hr']).sum() / proj['hr'].sum()

hr_falt = df_falt['hr'].sum()
print(f'IRA atual     : {ira_atual:.4f}   (HR {hr_atual})')
print(f'IRA projetado : {ira_projetado:.4f}   (HR {hr_atual + hr_falt}  | faltam {hr_falt} HR)')
print(f'Variacao      : {ira_projetado - ira_atual:+.4f}')
df_falt

IRA atual     : 7.7815   (HR 3165)
IRA projetado : 7.9823   (HR 3480  | faltam 315 HR)
Variacao      : +0.2008


,disciplina,nota,hr,tipo
0,Projeto de Componentes,10.0,60,faltante
1,Projeto de Refrigeracao e Climatizacao,10.0,60,faltante
2,Projeto de Transmissoes,10.0,60,faltante
3,Selecao de Materiais,10.0,30,faltante
4,Pratica de Termicas e Fluidos,10.0,30,faltante
5,Praticas de Sistemas Mecanicos,10.0,30,faltante
6,Pratica de Fabricacao,10.0,30,faltante
7,Implementacao e Operacao de Projeto Transformador,10.0,15,faltante


## Sensibilidade — e se todas as faltantes tivessem a mesma nota?

Mostra o IRA final se você tirasse a mesma nota em **todas** as disciplinas restantes.

In [5]:
conta = df[df['tipo'].isin(['cursada', 'reprovada'])]
num_atual = (conta['nota'] * conta['hr']).sum()   # numerador travado (ja cursado)
hr_falt = df_falt['hr'].sum()

print('Nota em TODAS as faltantes  ->  IRA final projetado')
for nota in [7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0]:
    ira = (num_atual + nota * hr_falt) / (hr_atual + hr_falt)
    print(f'        {nota:>4.1f}            ->        {ira:.4f}')

Nota em TODAS as faltantes  ->  IRA final projetado
         7.0            ->        7.6987
         7.5            ->        7.7517
         8.0            ->        7.8047
         8.5            ->        7.8576
         9.0            ->        7.9106
         9.5            ->        7.9636
        10.0            ->        8.0165
